# Практика · Тема 27 · HTTP-запити

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)

Наскрізний приклад той самий, що в лекції, — кавʼярня «Кома». Тільки тепер її меню
лежить не у файлі на диску, а на сервері, і по нього треба сходити.

Що зробимо:

1. піднімемо власний HTTP-сервер із меню кавʼярні;
2. зробимо перший `requests.get` і розберемо відповідь по частинах;
3. подивимось на заголовки й перевіримо `Content-Length` власними руками;
4. переконаємось, що `404` — це **успішно доставлена** відповідь, а не виняток;
5. відрізнимо `500` від `404` і подивимось, що робить `raise_for_status()`;
6. отримаємо справжню кракозябру й полагодимо її двома способами;
7. передамо параметр із пробілом, амперсандом і кирилицею через `params=`;
8. склеїмо той самий запит рядком і побачимо, як тихо псується результат;
9. напишемо власне відсоткове кодування й звіримо його з бібліотечним;
10. зловимо `Timeout` на маршруті, який навмисно спить;
11. зловимо `ConnectionError` там, де сервера немає взагалі;
12. пройдемо ланцюг перенаправлень і зазирнемо в `r.history`;
13. порахуємо зʼєднання: 40 запитів через `Session` і без неї;
14. приберемо за собою — зупинимо сервер.

> **Мережа не потрібна:** усі запити йдуть на наш власний сервер, який зошит піднімає
> сам на твоєму компʼютері. Інтернет для цього зошита не потрібен — потрібен лише
> встановлений пакет `requests`.

## 1 · Навчальний сервер — риштування

Наступна клітинка — **риштування**, а не матеріал теми.

Це наш навчальний сервер кавʼярні. Він написаний через **клас**, а класи будуть
у [темі 30](../30-classes/lecture.html) — зараз його не треба розуміти, просто запусти.
Читати варто хіба що перелік маршрутів у кінці клітинки: саме до них ми ходитимемо
весь зошит.

Два рядки в ньому все ж варто помітити, бо без них демонстрації брехали б:

* `protocol_version = "HTTP/1.1"` — інакше сервер закриває зʼєднання після кожної
  відповіді, і `Session` не має чого перевикористовувати;
* `disable_nagle_algorithm = True` — інакше кожна відповідь на постійному зʼєднанні
  чекає близько 40 мілісекунд через давню оптимізацію TCP, і «правильний» спосіб
  виглядав би повільнішим за неправильний.

`ThreadingHTTPServer(("127.0.0.1", 0), ...)` — адреса `127.0.0.1` означає «цей самий
компʼютер», а порт `0` означає «візьми будь-який вільний»: так зошит не посвариться
за порт із чимось іншим, що вже запущено.

In [ ]:
import json
import socket
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

import requests

MENU = [
    {"name": "Еспресо",  "price": 25, "category": "кава"},
    {"name": "Капучино", "price": 45, "category": "кава"},
    {"name": "Латте",    "price": 50, "category": "кава"},
    {"name": "Чай",      "price": 30, "category": "чай"},
]

# сюди сервер рахуватиме відкриті зʼєднання — знадобиться в розділі про Session
opened = {"connections": 0}


class CafeHandler(BaseHTTPRequestHandler):
    """Риштування: класи будуть у темі 30, зараз просто запусти."""

    protocol_version = "HTTP/1.1"        # без цього keep-alive не працює взагалі
    disable_nagle_algorithm = True       # без цього сервер бреше про швидкість

    def setup(self):
        # викликається рівно раз на кожне нове TCP-зʼєднання — так ми їх і рахуємо
        opened["connections"] += 1
        super().setup()

    def log_message(self, *args):
        # інакше зошит заросте рядками про кожен запит
        pass

    def reply(self, code, body, content_type, extra=None):
        data = body.encode("utf-8") if isinstance(body, str) else body
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.send_header("X-Cafe", "navchalnyi-server")
        for name, value in (extra or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        address = urlparse(self.path)
        path = address.path
        query = parse_qs(address.query)

        if path == "/":
            self.reply(200, "Кавʼярня «Кома». Меню: /menu\n", "text/plain; charset=utf-8")

        elif path == "/menu":
            rows = [f'{d["name"]} — {d["price"]} грн ({d["category"]})' for d in MENU]
            self.reply(200, "\n".join(rows) + "\n", "text/plain; charset=utf-8")

        elif path == "/menu.json":
            self.reply(200, json.dumps(MENU, ensure_ascii=False, indent=2),
                       "application/json; charset=utf-8")

        elif path == "/menu-no-charset":
            # той самий текст у UTF-8, але сервер «забув» сказати, у якому кодуванні
            rows = [f'{d["name"]} — {d["price"]} грн' for d in MENU]
            self.reply(200, "\n".join(rows) + "\n", "text/plain")

        elif path == "/search":
            word = query.get("q", [""])[0]
            found = [d for d in MENU
                     if word.lower() in (d["name"] + " " + d["category"]).lower()]
            names = ", ".join(d["name"] for d in found) if found else "нічого"
            self.reply(200, f"шукали: {word!r}\nзнайшли: {names}\n",
                       "text/plain; charset=utf-8")

        elif path == "/slow":
            time.sleep(1.5)              # навмисно повільний маршрут — для таймауту
            self.reply(200, "нарешті\n", "text/plain; charset=utf-8")

        elif path == "/broken":
            self.reply(500, "база даних не відповідає\n", "text/plain; charset=utf-8")

        elif path == "/old-menu":
            self.reply(301, "переїхали\n", "text/plain; charset=utf-8",
                       {"Location": "/menu-2019"})

        elif path == "/menu-2019":
            self.reply(301, "і ще раз переїхали\n", "text/plain; charset=utf-8",
                       {"Location": "/menu"})

        else:
            self.reply(404, f"немає сторінки {path}\n", "text/plain; charset=utf-8")


class QuietServer(ThreadingHTTPServer):
    """Той самий сервер, але мовчазний.

    Коли ми обірвемо запит по таймауту, сервер спробує відповісти в закрите
    зʼєднання й надрукує довгий traceback. Це нормальна ситуація, і в зошиті
    вона тільки заважає.
    """

    def handle_error(self, request, client_address):
        pass


server = QuietServer(("127.0.0.1", 0), CafeHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_address[1]}"

print("сервер працює на:", BASE)
print("маршрути:")
for route in ["/", "/menu", "/menu.json", "/menu-no-charset", "/search?q=…",
              "/slow", "/broken", "/old-menu", "будь-що інше → 404"]:
    print("   ", route)

## 2 · Перший запит

`requests.get(адреса, timeout=…)` робить рівно те, що описано в лекції: відкриває
зʼєднання, надсилає рядок запиту з заголовками, чекає на відповідь і повертає обʼєкт
`Response`. Розберемо його по частинах — код, пояснення до коду, тіло.

Таймаут ставимо **одразу**, а не «потім додамо»: запит без таймауту не має межі
очікування взагалі.

In [ ]:
r = requests.get(BASE + "/menu", timeout=5)

print("код стану :", r.status_code)      # число, за яким програма ухвалює рішення
print("пояснення :", r.reason)           # те саме словами, для людини
print("успіх?    :", r.ok)               # True, якщо код менший за 400
print("довжина   :", len(r.content), "байтів")
print("---- тіло відповіді ----")
print(r.text)

## 3 · Заголовки: службові дані про відповідь

`r.headers` поводиться як словник, але з двома відмінностями: регістр ключа не
має значення (`content-type` і `Content-Type` — те саме), і порядок задає сервер.

Заодно перевіримо сервер на чесність: заголовок `Content-Length` має точно збігатися
з кількістю байтів, які реально приїхали. Це наша перша перевірка «сказане проти
зробленого».

In [ ]:
for name, value in r.headers.items():
    print(f"{name:<16}: {value}")

promised_bytes = int(r.headers["Content-Length"])
received_bytes = len(r.content)
print()
print("обіцяно байтів :", promised_bytes)
print("приїхало байтів:", received_bytes)

assert promised_bytes == received_bytes, "сервер збрехав про довжину тіла!"
print("✅ Content-Length збігається з фактичною довжиною")

## 4 · `404` — це не помилка Python

Найважливіша пастка теми. Попросимо сторінку, якої немає.

Зверни увагу: жодного винятку. Запит пройшов ідеально — зʼєднання відкрилось,
сервер відповів, відповідь доїхала. Просто зміст відповіді — «такого немає».
Для Python це така сама вдала доставка, як і `200`.

In [ ]:
missing = requests.get(BASE + "/kavun", timeout=5)

print("код стану:", missing.status_code)
print("r.ok     :", missing.ok)
print("тіло     :", missing.text.strip())
print()
print("виняток не піднявся — програма спокійно дійшла до цього рядка")

Щоб `404` таки став винятком, про це треба попросити явно — `raise_for_status()`.
Метод мовчить на кодах до 400 і піднімає `requests.exceptions.HTTPError` на 4xx і 5xx.

Ловимо його так само, як будь-який інший виняток із [теми 19](../19-exceptions/lecture.html).

In [ ]:
try:
    missing.raise_for_status()
    print("сюди ми не потрапимо")
except requests.exceptions.HTTPError as error:
    print("HTTPError:", error)

# а на успішній відповіді той самий виклик просто мовчить і повертає None
print("на 200 raise_for_status() повертає:", r.raise_for_status())

## 5 · `500` — те саме «не 200», інша відповідальність

`404` і `500` обидва не є успіхом, але реагувати на них треба по-різному.

`404` каже: **ти** попросив те, чого немає. Повторювати запит безглуздо — відповідь
буде та сама. `500` каже: сервер зламався, обробляючи твій запит. Запит міг бути
бездоганний; через хвилину той самий запит може спрацювати.

In [ ]:
broken = requests.get(BASE + "/broken", timeout=5)

print("код стану:", broken.status_code)
print("пояснення:", broken.reason)
print("r.ok     :", broken.ok)
print("тіло     :", broken.text.strip())
print()

for response in [missing, broken]:
    family = response.status_code // 100
    advice = "повторювати безглуздо — виправ запит" if family == 4 else "має сенс повторити пізніше"
    print(f"{response.status_code}: {advice}")

## 6 · Звідки береться кракозябра

Мережею їдуть **байти**. Щоб перетворити їх на текст, треба знати кодування,
і називає його сервер — у заголовку `Content-Type` після слова `charset`.

Маршрут `/menu-no-charset` віддає той самий текст у UTF-8, але `charset` не називає.
За старим стандартом HTTP у такому разі текст вважається `ISO-8859-1` — саме це
припущення й перетворює українські літери на набір значків.

In [ ]:
no_charset = requests.get(BASE + "/menu-no-charset", timeout=5)

print("Content-Type :", no_charset.headers["Content-Type"])
print("r.encoding   :", no_charset.encoding, "← requests не мав звідки взяти краще")
print()
print("r.text (зіпсований):")
print(no_charset.text.split("\n")[0])
print()
print("r.content (сирі байти, перші 20):")
print(no_charset.content[:20])

Полагодити можна двома способами, і обидва варто знати:

1. розкодувати байти самому — `r.content.decode("utf-8")`;
2. підказати `requests` правильне кодування — присвоїти `r.encoding` і читати `r.text`
   знову; `r.text` рахується щоразу заново, тому нове значення одразу подіє.

In [ ]:
decoded_by_hand = no_charset.content.decode("utf-8")
print("1) через r.content.decode:", decoded_by_hand.split("\n")[0])

no_charset.encoding = "utf-8"          # підказали кодування — r.text перерахується
print("2) через r.encoding      :", no_charset.text.split("\n")[0])

assert decoded_by_hand == no_charset.text, "обидва способи мають дати той самий текст"
print("✅ обидва способи дають однаковий рядок")
print("а байти весь час були ті самі — псувалось лише тлумачення")

## 7 · Параметри запиту через `params=`

Пошук по меню приймає параметр `q`. Передамо в нього рядок, у якому є все, що
ламає ручне склеювання: пробіли, амперсанд і кирилиця.

`requests` сам перетворить кожен небезпечний символ на відсоткову послідовність.
Подивимось на `r.url` — це та адреса, яка реально пішла на сервер.

In [ ]:
query_text = "чай & кава"

clean = requests.get(BASE + "/search", params={"q": query_text}, timeout=5)

print("що ми просили    :", repr(query_text))
print("що пішло в мережу:")
print("  ", clean.url)
print()
print(clean.text)

## 8 · А тепер склеїмо руками

Той самий запит, зібраний конкатенацією рядків. Ніякої помилки, ніякого винятку —
просто інша відповідь.

Причина: символ `&` в адресі означає «а далі наступний параметр». Сервер побачив
не один параметр `q` зі значенням `чай & кава`, а параметр `q` зі значенням `чай `
і ще щось після амперсанда.

In [ ]:
glued_url = BASE + "/search?q=" + query_text
by_hand = requests.get(glued_url, timeout=5)

print("зібрана адреса:")
print("  ", glued_url)
print("що з неї вийшло в мережі:")
print("  ", by_hand.url)
print()
print(by_hand.text)

print("той самий запит дав різні відповіді:")
print("  через params= :", clean.text.split("\n")[1])
print("  склеєний      :", by_hand.text.split("\n")[1])

Найгірше тут не те, що відповідь інша, а те, що вона **правдоподібна**. Програма
не впала, у журналі чисто, у відповіді є дані — просто не ті. Такі помилки живуть
у коді роками.

## 9 · Наша реалізація проти бібліотечної

Перевіримо, що всередині немає магії. Напишемо відсоткове кодування самі:
беремо текст, перетворюємо його на байти UTF-8, і кожен байт, який не входить
до переліку безпечних, замінюємо на `%` і два шістнадцяткові знаки.

Потім звіримо результат із `urllib.parse.quote` — тією самою функцією зі
стандартної бібліотеки, на яку врешті спирається `requests`.

In [ ]:
from urllib.parse import quote

SAFE_SYMBOLS = ("ABCDEFGHIJKLMNOPQRSTUVWXYZ"
                "abcdefghijklmnopqrstuvwxyz"
                "0123456789-_.~")


def encode_value(text):
    """Наша власна версія відсоткового кодування — байт за байтом, без магії."""
    chunks = []
    for byte in text.encode("utf-8"):     # спершу текст → байти, і лише потім кодуємо
        symbol = chr(byte)
        if symbol in SAFE_SYMBOLS:
            chunks.append(symbol)
        else:
            chunks.append(f"%{byte:02X}")  # два шістнадцяткові знаки, великі літери
    return "".join(chunks)


for sample in ["latte", "чай & кава", "кава з молоком", "100%"]:
    ours = encode_value(sample)
    library = quote(sample, safe="")
    print(f"{sample!r:<20} → {ours}")
    assert ours == library, f"розійшлися на {sample!r}: {ours} проти {library}"

print()
print("✅ наша реалізація збігається з urllib.parse.quote на всіх пробах")
print("різниця лише в дрібниці: у рядку запиту requests пише пробіл як «+», а не «%20»")

## 10 · Таймаут

Маршрут `/slow` спить півтори секунди — це навмисно. На локальному сервері все
інше відповідає швидше за мілісекунду, і поставити таймаут, який спрацює, просто
ніде: запит завжди встигає.

Спершу дамо півсекунди й не дочекаємось. Потім дамо три — і дочекаємось.

In [ ]:
started = time.perf_counter()
try:
    requests.get(BASE + "/slow", timeout=0.5)
    print("сюди ми не потрапимо")
except requests.exceptions.Timeout as error:
    elapsed_ms = (time.perf_counter() - started) * 1000
    print(f"не дочекались за {elapsed_ms:.0f} мс")
    print("тип винятку:", type(error).__name__, "— підвид Timeout")

started = time.perf_counter()
patient = requests.get(BASE + "/slow", timeout=3)
elapsed_ms = (time.perf_counter() - started) * 1000
print(f"з timeout=3 дочекались: {patient.status_code} за {elapsed_ms:.0f} мс")

Зверни увагу на два числа: перший запит обірвався приблизно на 500 мс, другий
дочекався приблизно на 1500 мс. Таймаут — це **межа очікування**, а не час запиту.

## 11 · Помилка мережі — це інший сорт біди

`404` і `500` приїхали **від сервера**. А тепер постукаємо туди, де сервера немає
взагалі: візьмемо вільний порт і одразу його звільнимо.

Тут не буде жодного коду стану — бо не буде відповіді. Буде `ConnectionError`.

In [ ]:
# просимо систему дати вільний порт і одразу віддаємо його назад:
# так ми майже напевно стукаємо туди, де ніхто не слухає
probe = socket.socket()
probe.bind(("127.0.0.1", 0))
free_port = probe.getsockname()[1]
probe.close()

try:
    requests.get(f"http://127.0.0.1:{free_port}/menu", timeout=2)
    print("сюди ми не потрапимо")
except requests.exceptions.ConnectionError as error:
    print("ConnectionError — відповіді немає взагалі")
    print("перші 90 символів повідомлення:")
    print("  ", str(error)[:90])

print()
print("порівняй три випадки:")
print("  сервера немає   → ConnectionError, коду стану немає")
print(f"  сервер зламався → код {broken.status_code}, відповідь є")
print(f"  сторінки немає  → код {missing.status_code}, відповідь є")

## 12 · Перенаправлення

Меню в нашій кавʼярні переїжджало двічі: `/old-menu` веде на `/menu-2019`,
а той — на `/menu`. `requests` за замовчуванням іде за перенаправленнями сам,
тому ми одразу отримуємо кінцеву сторінку.

Слід не зникає: проміжні відповіді лежать у `r.history`, а кінцева адреса — в `r.url`.

In [ ]:
final = requests.get(BASE + "/old-menu", timeout=5)

print("просили       :", BASE + "/old-menu")
print("отримали код  :", final.status_code)
print("кінцева адреса:", final.url)
print("довжина history:", len(final.history))
for hop in final.history:
    print(f"   {hop.status_code} {hop.url}  →  {hop.headers['Location']}")

assert final.url.endswith("/menu"), "мали доїхати до /menu"
print("✅ доїхали куди треба, зробивши", len(final.history) + 1, "запити")

А тепер попросимо `requests` **не** ходити за перенаправленням. Тоді ми побачимо
саме ту відповідь, яку віддав перший сервер: код `301` і заголовок `Location`.
Це те, що бачить браузер перед тим, як вирішити перейти.

In [ ]:
first_hop = requests.get(BASE + "/old-menu", timeout=5, allow_redirects=False)

print("код стану :", first_hop.status_code)
print("Location  :", first_hop.headers["Location"])
print("r.url     :", first_hop.url, "← адреса не змінилась")
print("r.history :", first_hop.history, "← порожня, ми нікуди не ходили")
print("r.ok      :", first_hop.ok, "← 301 менший за 400, тож формально успіх")

## 13 · `Session`: одне зʼєднання замість сорока

Головний вимір теми. Зробимо 40 запитів двома способами й порахуємо не час,
а **зʼєднання** — наш сервер рахує їх сам, у лічильнику `opened`.

Час теж покажемо, але на нього дивись обережно: на локальній петлі зʼєднання
дуже дешеве, і числа стрибають від запуску до запуску. Кількість зʼєднань
не стрибає ніколи.

In [ ]:
HOW_MANY = 40

opened["connections"] = 0
started = time.perf_counter()
with requests.Session() as session:
    for _ in range(HOW_MANY):
        session.get(BASE + "/menu", timeout=5)
time_with_session = (time.perf_counter() - started) * 1000
conns_with_session = opened["connections"]

opened["connections"] = 0
started = time.perf_counter()
for _ in range(HOW_MANY):
    requests.get(BASE + "/menu", timeout=5)
time_without_session = (time.perf_counter() - started) * 1000
conns_without_session = opened["connections"]

print(f"із Session : {conns_with_session:>3} зʼєднань, {time_with_session:6.0f} мс")
print(f"без Session: {conns_without_session:>3} зʼєднань, {time_without_session:6.0f} мс")

assert conns_with_session == 1, "Session мала обійтися одним зʼєднанням"
assert conns_without_session == HOW_MANY, "без Session кожен запит відкриває своє зʼєднання"
print("✅ 1 зʼєднання проти", HOW_MANY, "— і це число не залежить від машини")

На нашій петлі виграш у часі невеликий: зʼєднатися з сусіднім процесом дешево.
У справжній мережі до кожного такого зʼєднання додається подорож пакета туди-назад
і рукостискання TLS — там та сама заміна дає вже не відсотки, а рази.

Ще одна користь `Session`, яку видно тут же: спільні заголовки задаються один раз
і їдуть з кожним запитом.

In [ ]:
with requests.Session() as session:
    session.headers.update({"X-Client": "python-lessons-27"})
    first = session.get(BASE + "/menu", timeout=5)
    second = session.get(BASE + "/search", params={"q": "кава"}, timeout=5)

print("заголовок першого запиту :", first.request.headers["X-Client"])
print("заголовок другого запиту :", second.request.headers["X-Client"])
print()
print("відповідь другого запиту:")
print(second.text)

## 14 · Текст для очей і дані для програм

Наш сервер має ще один маршрут — `/menu.json`. Те саме меню, але у форматі
для програм. Ми вміємо його **принести**; перетворити на список словників
і працювати з ним — це вже тема 28, де для цього є один короткий виклик `r.json()`.

In [ ]:
for_machine = requests.get(BASE + "/menu.json", timeout=5)

print("Content-Type:", for_machine.headers["Content-Type"])
print("перші рядки тіла:")
print("\n".join(for_machine.text.split("\n")[:5]))
print("...")
print()
print("тип того, що ми маємо зараз:", type(for_machine.text).__name__)

## 15 · Прибираємо за собою

Сервер живе у фоновому потоці. Поки зошит працює, це нормально, але лишати його
висіти після завершення не варто — так само, як [тема 22](../22-csv-json/lecture.html)
прибирала за собою тимчасову теку.

`shutdown()` просить цикл обробки зупинитись, `server_close()` звільняє порт.

In [ ]:
server.shutdown()
server.server_close()
print("сервер зупинено, порт", server.server_address[1], "звільнено")

# переконаємось, що він справді більше не відповідає
try:
    requests.get(BASE + "/menu", timeout=2)
    print("а він усе ще відповідає — щось пішло не так")
except requests.exceptions.ConnectionError:
    print("✅ ConnectionError — сервера більше немає, як і має бути")

## Завдання

Три рівні. Повний розбір із критеріями «зроблено» — у [homework.html](homework.html).

### 🟢 Рівень 1
Підніми сервер знову (виконай клітинку з розділу 1 ще раз) і додай у `do_GET`
маршрут `/hours`, який віддає години роботи кавʼярні звичайним текстом.
Сходи по нього й надрукуй код стану та тіло.

### 🟡 Рівень 2
Напиши функцію `fetch(url)`, яка повертає текст відповіді, а на кодах `4xx` і `5xx`
повертає рядок із поясненням замість тексту — **без** винятку назовні.
Перевір її на `/menu`, `/kavun` і `/broken`.

### 🔴 Рівень 3
Додай маршрут `/flaky`, який віддає `503` перші два рази й `200` на третій
(лічильник тримай у словнику, як `opened`). Напиши цикл, який повторює запит
до трьох разів і зупиняється на першій вдалій відповіді — і не повторює
на `404`.